# F1 Lap Time Predictor

Predicts lap times within a single Formula 1 race using tire age, comparing a baseline (grid position + lap number) against models that also know tire age.

**Setup:** place `races.csv`, `lap_times.csv`, `pit_stops.csv`, `results.csv` in this folder, then Run All.

## Race and drivers

Analysis is performed on the **2023 Italian Grand Prix (`raceId = 1112`) at Monza**. Weather reports and lap data confirm this race was fully dry with zero red flags or safety car periods to distort green-flag pace. The top 8 classified drivers who completed full race distance (`statusId = 1`) were selected to ensure clean, full stint data.

In [ ]:
import pandas as pd
import numpy as np

races = pd.read_csv('races.csv')
lap_times = pd.read_csv('lap_times.csv')
pit_stops = pd.read_csv('pit_stops.csv')
results = pd.read_csv('results.csv')

RACE_ID = 1112  # 2023 Italian Grand Prix (Monza)

df_laps = lap_times[lap_times['raceId'] == RACE_ID].copy()
df_pits = pit_stops[pit_stops['raceId'] == RACE_ID].copy()
df_res = results[results['raceId'] == RACE_ID].copy()

df_laps['lap_time_sec'] = df_laps['milliseconds'] / 1000.0

finishers = df_res[df_res['statusId'] == 1]['driverId'].unique()[:8]
df_laps = df_laps[df_laps['driverId'].isin(finishers)].copy()
df_laps = df_laps.merge(df_res[['driverId', 'grid']], on='driverId', how='left')

print(f"Loaded {len(df_laps)} raw laps for {len(finishers)} drivers.")

## Cleaning

Data cleaning removes two main artifacts:
1. **Pit laps:** in-laps (entering pit lane) and immediate out-laps (exiting pit lane) were removed per driver to eliminate non-racing pace artifacts.
2. **Pace outliers:** laps exceeding 1.5x the driver's median lap time (e.g. blue flag delays) were removed.

A total of **16 outlier laps** were removed across the selected driver set.

In [ ]:
pit_pairs = set(zip(df_pits['driverId'], df_pits['lap']))
pit_and_after_pairs = pit_pairs | {(d, l + 1) for d, l in pit_pairs}

initial_count = len(df_laps)
is_pit_or_after = df_laps.apply(lambda r: (r['driverId'], r['lap']) in pit_and_after_pairs, axis=1)
cleaned_df = df_laps[~is_pit_or_after].copy()

driver_medians = cleaned_df.groupby('driverId')['lap_time_sec'].transform('median')
cleaned_df = cleaned_df[cleaned_df['lap_time_sec'] <= 1.5 * driver_medians].copy()

removed_count = initial_count - len(cleaned_df)
print(f"Raw laps: {initial_count}")
print(f"Removed: {removed_count}")
print(f"Remaining: {len(cleaned_df)}")

## Feature: tire age

`tire_age` represents the number of laps completed on the current set of tires. It is calculated via direct subtraction relative to the preceding pit stop lap (`lap - stint_start`), ensuring it stays accurate and resets to zero after every pit stop regardless of any dropped or outlier laps.

In [ ]:
def add_stint_features(df, pit_stops_df):
    pieces = []
    for driver_id, group in df.groupby('driverId'):
        group = group.sort_values('lap').copy()
        driver_pits = sorted(pit_stops_df[pit_stops_df['driverId'] == driver_id]['lap'].tolist())
        stint_bounds = [0] + driver_pits + [group['lap'].max()]
        lap_vals = group['lap'].values
        stint = np.zeros(len(group), dtype=int)
        stint_start = np.zeros(len(group), dtype=int)
        for i in range(len(stint_bounds) - 1):
            lo, hi = stint_bounds[i], stint_bounds[i + 1]
            mask = (lap_vals > lo) & (lap_vals <= hi)
            stint[mask] = i + 1
            stint_start[mask] = lo + 1
        group['stint'] = stint
        group['tire_age'] = group['lap'] - stint_start
        pieces.append(group)
    return pd.concat(pieces, ignore_index=True)

cleaned_df = add_stint_features(cleaned_df, df_pits)
print(cleaned_df[['driverId', 'lap', 'stint', 'tire_age']].head(10).to_string(index=False))

## Split: stint-based, not random

To avoid data leakage, data is split temporally by stint rather than using a random split. Each driver's **final stint** is held out as the test set, while all earlier stints form the training set. This tests the model's ability to forecast performance on genuinely unseen tire ages, rather than interpolating between adjacent laps a random split would leak across train and test.

In [ ]:
max_stints = cleaned_df.groupby('driverId')['stint'].transform('max')
train_df = cleaned_df[cleaned_df['stint'] < max_stints].copy()
test_df = cleaned_df[cleaned_df['stint'] == max_stints].copy()

print(f"Training laps: {len(train_df)}")
print(f"Test laps (held-out final stints): {len(test_df)}")

## Models compared

- **Baseline** — Linear Regression on `grid` + `lap`.
- **Linear Regression (+Tire Age)** — adds `tire_age` to the baseline features.
- **Random Forest (+Tire Age)** — non-linear regressor, `n_estimators=100`, `random_state=42` for reproducibility.

**Note on feature disambiguation:** within a single stint, `lap` and `tire_age` increase together almost linearly. What separates them is that training data spans multiple stints — the same `tire_age` value recurs at different absolute lap numbers across different stints, which is what lets the model attribute variance to tire age specifically rather than lap number alone.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

baseline_features = ['grid', 'lap']
enhanced_features = ['grid', 'lap', 'tire_age']
target = 'lap_time_sec'

lr_base = LinearRegression().fit(train_df[baseline_features], train_df[target])
preds_base = lr_base.predict(test_df[baseline_features])

lr_enh = LinearRegression().fit(train_df[enhanced_features], train_df[target])
preds_lr_enh = lr_enh.predict(test_df[enhanced_features])

rf_enh = RandomForestRegressor(n_estimators=100, random_state=42)
rf_enh.fit(train_df[enhanced_features], train_df[target])
preds_rf_enh = rf_enh.predict(test_df[enhanced_features])

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return round(rmse, 3), round(mae, 3)

comp = pd.DataFrame([
    {"Model": "Linear Regression (Baseline: grid+lap)", "RMSE": evaluate(test_df[target], preds_base)[0], "MAE": evaluate(test_df[target], preds_base)[1]},
    {"Model": "Linear Regression (+Tire Age)", "RMSE": evaluate(test_df[target], preds_lr_enh)[0], "MAE": evaluate(test_df[target], preds_lr_enh)[1]},
    {"Model": "Random Forest (+Tire Age)", "RMSE": evaluate(test_df[target], preds_rf_enh)[0], "MAE": evaluate(test_df[target], preds_rf_enh)[1]},
])
print(comp)

## Predicted vs actual — one full held-out stint

In [ ]:
import matplotlib.pyplot as plt

test_df = test_df.reset_index(drop=True)
test_df['pred_rf_enh'] = preds_rf_enh
sample_driver = test_df['driverId'].iloc[0]
driver_data = test_df[test_df['driverId'] == sample_driver].sort_values('lap')

plt.figure(figsize=(10, 5))
plt.plot(driver_data['lap'], driver_data['lap_time_sec'], label='Actual Lap Time', marker='o')
plt.plot(driver_data['lap'], driver_data['pred_rf_enh'], label='Predicted Lap Time (RF)', linestyle='--', color='orange')
plt.xlabel('Lap Number')
plt.ylabel('Lap Time (seconds)')
plt.title(f'Driver ID {sample_driver} Final Stint: Actual vs. Predicted Lap Time')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('stint_visualization.png')
plt.show()

## Conclusion

Adding `tire_age` gives a small, measurable improvement over the baseline for Linear Regression (RMSE 0.858 → 0.849, roughly a 1% reduction), a real but modest effect, consistent with Monza being one of the lowest tire wear circuits on the calendar. Random Forest performs slightly worse than the baseline (0.931), likely because it has enough flexibility to fit noise in a fairly small dataset (a handful of drivers, one race) rather than a genuine non-linear degradation curve.

**Key limitations:**
1. The model does not explicitly account for fuel-burn weight reduction, which offsets tire degradation over time.
2. Compound types (Soft vs. Medium vs. Hard) are not differentiated in the standard timing data.